# Plant Disease Identification using MobileNetV2 + ECA Attention

This notebook trains a plant disease classification model using:
- **MobileNetV2** as the backbone (pretrained on ImageNet)
- **ECA (Efficient Channel Attention)** layer
- **PlantVillage** dataset (~40K images, 16 classes)

---

## Step 1: Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================
# SET YOUR DATASET PATH HERE
# ============================================
# Change this to match where you uploaded the PlantVillage folder in your Drive.
# For example:
#   My Drive / PlantVillage / Tomato_healthy / ...  -->  "/content/drive/MyDrive/PlantVillage"
#   My Drive / plant-disease / PlantVillage / ...   -->  "/content/drive/MyDrive/plant-disease/PlantVillage"

DATASET_DIR = "/content/drive/MyDrive/plant_disease_project/PlantVillage"

# Verify the path is correct
import os
if os.path.exists(DATASET_DIR):
    classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)) and not d.startswith('.')])
    print(f"Dataset found! {len(classes)} classes detected:")
    for c in classes:
        count = len(os.listdir(os.path.join(DATASET_DIR, c)))
        print(f"  {c}: {count} images")
else:
    print(f"ERROR: Path not found: {DATASET_DIR}")
    print("\nListing your Drive root to help you find it:")
    for item in os.listdir('/content/drive/MyDrive/'):
        print(f"  {item}")

In [ ]:
# Check GPU availability
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

if not tf.config.list_physical_devices('GPU'):
    print("\nWARNING: No GPU detected!")
    print("Go to: Runtime > Change runtime type > Hardware accelerator > GPU (T4)")

## Step 2: Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import os
import shutil

## Step 3: Load Dataset

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 24

# Remove hidden folders (like .ipynb_checkpoints) if they exist
for folder in os.listdir(DATASET_DIR):
    folder_path = os.path.join(DATASET_DIR, folder)
    if folder.startswith(".") and os.path.isdir(folder_path):
        shutil.move(folder_path, folder_path + "_backup")
        print(f"Moved hidden folder: {folder}")

# Training dataset (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Validation dataset (20%)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nClasses detected ({num_classes}):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## Step 4: Preview Sample Images

In [ ]:
plt.figure(figsize=(15, 10))
for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle("Sample Training Images", fontsize=16)
plt.tight_layout()
plt.show()

## Step 5: Define ECA (Efficient Channel Attention) Layer

In [ ]:
class ECALayer(layers.Layer):
    def __init__(self, gamma=2, b=1, **kwargs):
        super(ECALayer, self).__init__(**kwargs)
        self.gamma = gamma
        self.b = b

    def build(self, input_shape):
        channels = input_shape[-1]
        t = int(abs((tf.math.log(tf.cast(channels, tf.float32)) / tf.math.log(2.0)) + self.b) / self.gamma)
        k = t if t % 2 else t + 1
        self.avg_pool = layers.GlobalAveragePooling2D(keepdims=True)
        self.conv = layers.Conv1D(1, kernel_size=k, padding="same", use_bias=False)
        self.sigmoid = layers.Activation("sigmoid")

    def call(self, x):
        y = self.avg_pool(x)
        y = tf.squeeze(y, axis=[1, 2])
        y = tf.expand_dims(y, axis=-1)
        y = self.conv(y)
        y = self.sigmoid(y)
        y = tf.reshape(y, [-1, 1, 1, x.shape[-1]])
        return x * y

    def get_config(self):
        config = super(ECALayer, self).get_config()
        config.update({
            "gamma": self.gamma,
            "b": self.b
        })
        return config

## Step 6: Build Model (MobileNetV2 + ECA)

In [ ]:
def build_mobilenetv2_eca(input_shape=(224, 224, 3), num_classes=16):
    base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    base_model.trainable = True  # Unfreeze backbone for fine-tuning

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=True)

    # ECA Attention
    x = ECALayer()(x)

    # Extra Conv Layers
    x = layers.Conv2D(256, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    # Global Average Pooling
    x = layers.GlobalAveragePooling2D()(x)

    # Dense Layers
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    # Output
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = build_mobilenetv2_eca(input_shape=(224, 224, 3), num_classes=num_classes)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Step 7: Train the Model

In [ ]:
EPOCHS = 50  # Adjust as needed

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

## Step 8: Training History Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Step 9: Evaluation - Confusion Matrix & Classification Report

In [ ]:
# Collect predictions
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

# Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Classification Report
all_labels = list(range(num_classes))
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    y_true,
    y_pred,
    labels=all_labels,
    target_names=class_names,
    zero_division=0,
    digits=4
))

In [ ]:
!pip install seaborn -q

In [ ]:
import seaborn as sns

plt.figure(figsize=(20, 18))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted", fontsize=14)
plt.ylabel("True", fontsize=14)
plt.title("Normalized Confusion Matrix", fontsize=16)
plt.tight_layout()
plt.show()

## Step 10: Save Model

In [ ]:
# Save to Google Drive so it persists after Colab disconnects
SAVE_PATH = "/content/drive/MyDrive/mobilenetv2_eca_plant_disease.h5"
model.save(SAVE_PATH)
print(f"Model saved to: {SAVE_PATH}")